In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from harbor.analysis.cross_docking import DockingDataModel
import plotting_params as p
from importlib import reload
reload(p)

## input files

In [ ]:
posit_raw = DockingDataModel.deserialize("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ALL_combined_results.parquet")

In [ ]:
posit_results = Path("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/all_evals_posit_combined_results.csv")

In [ ]:
pdf = pd.read_csv(posit_results)
pdf["Error_Lower"] = pdf["Fraction"] - pdf["CI_Lower"]
pdf["Error_Lower"] = pdf["Error_Lower"].apply(lambda x: 0 if x < 0 else x)
pdf["Error_Upper"] = pdf["CI_Upper"] - pdf["Fraction"]
pdf["Error_Upper"] = pdf["Error_Upper"].apply(lambda x: 0 if x < 0 else x)

In [ ]:
fdf = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/all_evals_fred_combined_results.csv")

## output files

In [ ]:
figpath = Path("../figures")
figpath.mkdir(exist_ok=True)

# Reference Split Comparison

In [ ]:
df = pdf[(~pdf["Reference_Split"].isna())&(pdf["PairwiseSplit"].isna())]

In [ ]:
df = df.groupby(["Reference_Split", "Score", "N_Reference_Structures"]).head(1)

In [ ]:
ALPHA = 0.3
p.figure_decorator(func=p.plot_filled_in_error_bars, label_map=p.label_map, fig_path=figpath / "dataset_split", raw_df=df)
ALPHA = 0.2

In [ ]:
def plot_filled_in_error_bars_facet_col(
    raw_df,
    x_var=X_VAR,
    y_var=Y_VAR,
    color_var=COLOR_VAR,
    style_var=STYLE_VAR,
    facet_var=STYLE_VAR,  # New parameter, defaults to None
    ci_lower=CI_LOWER,
    ci_upper=CI_UPPER,
):
    """Plot filled-in error bars with facets by specified variable"""
    # Use style_var as facet_var if none provided
    facet_var = facet_var or style_var
    style_var = style_var or color_var

    # Sort the dataframe
    raw_df = raw_df.sort_values(by=[x_var, style_var, color_var])

    # Create subplot for each facet value
    facets = raw_df[facet_var].unique()
    fig, axes = plt.subplots(1, len(facets), figsize=(LARGE_FIG_SIZE[0]*len(facets), LARGE_FIG_SIZE[1]))

    # Create color mapping
    unique_colors = sns.color_palette(n_colors=len(raw_df[color_var].unique()))
    color_map = dict(zip(sorted(raw_df[color_var].unique()), unique_colors))

    for ax, facet in zip(axes, facets):
        facet_data = raw_df[raw_df[facet_var] == facet]

        # Create fill between for each group using matched colors
        for name, group in facet_data.groupby([color_var, style_var]):
            color_name = name[0]  # First element is Score
            ax.fill_between(
                group[x_var],
                group[ci_lower],
                group[ci_upper],
                color=color_map[color_name],
                alpha=ALPHA,
            )

        # Create the line plot
        sns.lineplot(
            data=facet_data,
            x=x_var,
            y=y_var,
            hue=color_var,
            style=style_var,  # Keep style_var for line styles
            ax=ax,
            palette=color_map,
            hue_order=list(reversed(sorted(raw_df[color_var].unique()))),
            style_order=list(reversed(sorted(raw_df[style_var].unique())))
        )
        
        # Customize each subplot
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(ScalarFormatter())

        custom_ticks = [1, 5, 10, 20, 50, 100, 200, raw_df[x_var].max()]
        ax.set_xticks(custom_ticks)
        ax.set_xticklabels(custom_ticks, fontsize=FONT_SIZES["ticks"])
        ax.tick_params(axis='y', labelsize=FONT_SIZES["ticks"])

        ax.set_xlabel(X_LABEL, fontsize=FONT_SIZES["xlabel"], fontweight="bold")
        if ax == axes[0]:  # Only set ylabel for first subplot
            ax.set_ylabel(Y_LABEL, fontsize=FONT_SIZES["ylabel"], fontweight="bold")
        else:
            ax.set_ylabel("")

        ax.set_title(f"{facet}", fontsize=FONT_SIZES["xlabel"], fontweight="bold")

        # Customize legend
        legend = ax.legend()
        plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
        plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])
    return plt

In [ ]:
figure_decorator(plot_filled_in_error_bars_facet_col,color_var=STYLE_VAR, label_map=label_map,facet_var=COLOR_VAR, raw_df=df, fig_path=figpath / "dataset_split_wide")

### is RMSD datesplit actually better than RandomSplit?

In [ ]:
from harbor.analysis.cross_docking import EvaluatorFactory, Results, Evaluator

In [ ]:
evf = EvaluatorFactory(name="test")
evf.reference_split_settings.use = True
evf.reference_split_settings.date_split_settings.use = True
evf.reference_split_settings.date_split_settings.reference_structure_date_column = (
    "Reference_Structure_Date"
)
evf.reference_split_settings.random_split_settings.use = True
evf.scorer_settings.rmsd_scorer_settings.use = True
evf.scorer_settings.posit_scorer_settings.use = False

evf.reference_split_settings.n_reference_structures = [100]
evf.n_bootstraps = 1
evs: [Evaluator] = evf.create_evaluators(posit_raw)

In [ ]:
random = evs[0]
date = evs[1]
raw = random.run_pose_selector(posit_raw)

In [ ]:
testdf = raw.dataframe
testdf = testdf.sort_values("RMSD", ascending=True).groupby(["Query_Ligand"]).head(1)

In [ ]:
test_scoring = DockingDataModel(dataframe=testdf, **raw.model_dump())

In [ ]:
scored = random.calculate_results([test_scoring])

In [ ]:
evf.reference_split_settings.n_reference_structures = [100]
evf.n_bootstraps = 1
evs: [Evaluator] = evf.create_evaluators(posit_raw)
random = evs[0]
date = evs[1]
rdf = random.run_dataset_split(raw)[0].dataframe
ddf = date.run_dataset_split(raw)[0].dataframe

simdf = pd.concat([pd.DataFrame({"Split": "RandomSplit", "Max MCS Tanimoto":rdf[rdf["Type"] == "MCS"].groupby("Query_Ligand")["Tanimoto"].max(), "Min MCS Tanimoto":rdf[rdf["Type"] == "MCS"].groupby("Query_Ligand")["Tanimoto"].min()}),
                   pd.DataFrame({"Split": "DateSplit", "Max MCS Tanimoto":ddf[ddf["Type"] == "MCS"].groupby("Query_Ligand")["Tanimoto"].max(), "Min MCS Tanimoto":ddf[ddf["Type"] == "MCS"].groupby("Query_Ligand")["Tanimoto"].min()}),
                  ])

In [ ]:
from scipy import stats
import numpy as np
def plot_filled_ecdf_minmax(data, x_min_column, x_max_column, hue_column, complementary=True, alpha=0.2):
    # Get unique values for the hue column
    hue_values = data[hue_column].unique()

    # Create figure
    plt.figure(figsize=SMALL_FIG_SIZE)

    ecdfs = {}
    # Calculate ECDFs for both min and max for each split
    for hue in hue_values:
        subset_min = data[data[hue_column] == hue][x_min_column]
        subset_max = data[data[hue_column] == hue][x_max_column]
        
        # Calculate ECDFs
        ecdf_min = stats.ecdf(subset_min)
        ecdf_max = stats.ecdf(subset_max)
        
        x_min = np.sort(subset_min)
        x_max = np.sort(subset_max)
        
        y_min = ecdf_min.cdf.evaluate(x_min)
        y_max = ecdf_max.cdf.evaluate(x_max)

        if complementary:
            y_min = 1 - y_min
            y_max = 1 - y_max

        ecdfs[hue] = {
            'min': (x_min, y_min),
            'max': (x_max, y_max)
        }

    # Plot for each split type
    for hue in hue_values:
        x_min, y_min = ecdfs[hue]['min']
        x_max, y_max = ecdfs[hue]['max']
        
        # Create common x values for fill_between
        x_all = np.unique(np.concatenate([x_min, x_max]))
        y_min_interp = np.interp(x_all, x_min, y_min)
        y_max_interp = np.interp(x_all, x_max, y_max)

        # Fill between min and max curves
        plt.fill_between(x_all, y_min_interp, y_max_interp, alpha=alpha, label=hue)

        # Plot the boundary lines
        # plt.plot(x_min, y_min, color=plt.gca().lines[-1].get_color())
        # plt.plot(x_max, y_max, color=plt.gca().lines[-1].get_color())

    plt.xlabel("MCS Tanimoto")
    plt.ylabel("ECDF")
    plt.legend()
    return plt

In [ ]:
plt = plot_filled_ecdf_minmax(simdf,
                       x_min_column="Min MCS Tanimoto",
                       x_max_column="Max MCS Tanimoto",
                       hue_column="Split",
                       complementary=True,
                              alpha=0.5)
plt = update_labels(plt, label_map, x_label="MCS Tanimoto Range to the References", y_label="Fraction of Molecules", legend_title="Reference Split")
save_figure(plt, figpath / "mcs_tanimoto_range")

In [ ]:
rdf = random.run_dataset_split(raw)[0].dataframe
testdf = rdf.sort_values("RMSD", ascending=True).groupby(["Query_Ligand"]).head(1)
test_scoring = DockingDataModel(dataframe=testdf, **raw.model_dump())
scored = random.calculate_results([test_scoring])
print(scored)

In [ ]:
ddf = date.run_dataset_split(raw)[0].dataframe
testdf = ddf.sort_values("RMSD", ascending=True).groupby(["Query_Ligand"]).head(1)
test_scoring = DockingDataModel(dataframe=testdf, **raw.model_dump())
scored = random.calculate_results([test_scoring])
print(scored)

In [ ]:
results = Results.df_from_results(Results.calculate_results(posit_raw, evs))

In [ ]:
results

In [ ]:
prdf = posit_raw.dataframe

In [ ]:
refs = prdf.Reference_Structure.unique()[:100]

In [ ]:
prdf = prdf[prdf["Reference_Structure"].isin(refs)]

# Scaffold Split

In [ ]:
sdf = pdf[(pdf["PairwiseSplit"] == "ScaffoldSplit")&(pdf["Scaffold_Split_Option"].isin(['x_to_y', 'x_to_x']))&(pdf["Reference_Split"].isna())]

In [ ]:
sdf.nunique()

In [ ]:
# replace brackets in the query and ref columns
query = "Query_Scaffold_ID_Subset"
ref = "Reference_Scaffold_ID_Subset"
sdf[query] = (
    sdf[query]
    .astype(str)
    .apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
)
sdf["qint"] = sdf[query].astype(float).tolist()
sdf[ref] = (
    sdf[ref]
    .astype(str)
    .apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
)
sdf["rint"] = sdf[ref].astype(float).tolist()
sdf = sdf.groupby(["Score", "qint", 'rint']).head(1)

In [ ]:
sdf.sort_values(["Score", "rint"]).groupby(["rint"]).head(1)[["rint", "Total"]]

In [ ]:
sdf.sort_values(["Score", "qint"]).groupby(["qint"]).head(1)[["qint", "Total"]]

## Scaffold x_to_y heatmap

In [ ]:
heatmap_dfs = {
        "_".join(name): group for name, group in sdf.groupby(["Score"])
    }

In [ ]:
def get_label(var):
    return label_map.get(var,var)

In [ ]:
fig_name = "scaffold_x_to_y_heatmap"
for name, heatmap_df in heatmap_dfs.items():
    pivot_fraction = heatmap_df.pivot(
        index="qint", columns="rint", values="Fraction"
    )
    ref_counts = (
        heatmap_df.sort_values("rint")
        .groupby(ref)
        .head(1)[[ref, "Total"]]
        .to_dict(orient="records")
    )
    count_dict = {data[ref]: data["Total"] for data in ref_counts}

    query_counts = (
        heatmap_df.sort_values("qint")
        .groupby(query)
        .head(1)[[query, "Total"]]
        .to_dict(orient="records")
    )
    count_dict = {data[query]: data["Total"] for data in query_counts}
    
    ytick_labels = [
        f"$\\bf{int(cluster_id) + 1}$ ({total})" for cluster_id, total in count_dict.items()
    ]
    xtick_labels = [
        f"$\\bf{int(cluster_id) + 1}$\n({total})" for cluster_id, total in count_dict.items()
    ]
    plt.figure(figsize=LARGE_FIG_SIZE)
    
    # Create custom annotation array
    annotations = pivot_fraction.copy()
    annotations = annotations.map(lambda x: '' if x in [0.0] else f'{x:.1f}')
    heatmap = sns.heatmap(
        data=pivot_fraction,
        xticklabels=xtick_labels,
        yticklabels=ytick_labels,
        annot=annotations,
        fmt='',
        cmap="coolwarm_r",
        vmin=0, 
        vmax=1
    )
    # Add colorbar label
    heatmap.collections[0].colorbar.set_label("Fraction")

    # Rotate axis labels for better readability
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)

    # Invert y-axis to put 0 at bottom
    plt.gca().invert_yaxis()

    # Set axis labels
    plt.xlabel(
        f"$\\bf{{Reference Scaffold ID}}$\n (# Reference Structures with Scaffold)",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="normal",
    )
    plt.ylabel(
        f"$\\bf{{Query Scaffold ID}}$\n (# Query Ligands with Scaffold)",
        fontsize=FONT_SIZES["ylabel"],
        fontweight="normal",
    )
    plt.title(
        f"Scored by {get_label(name)}",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="bold",
    )

    plt.savefig(figpath / f"{fig_name}_{name}.svg", format="svg", bbox_inches="tight")
    plt.savefig(figpath / f"{fig_name}_{name}.png", format="png", bbox_inches="tight")

### sort by size of scaffold

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

def draw_scaffold(mol, output_path, size=(400, 400), bond_length=30):
    # Set drawing options
    opts = Draw.DrawingOptions()
    opts.bondLength = bond_length  # Fixed bond length for all molecules
    opts.fixedBondLength = bond_length
    opts.coordScale = 1.0
    
    # Draw the molecule with consistent bond length
    img = Draw.MolToImage(
        mol,
        size=size,
        options=opts,
        # kekulize=True,
        # fitImage=True,    # Auto-scales to fit the image size while maintaining bond length ratio
        imageType="png"
    )
    
    img.save(output_path)

def get_scaffold_size(mol):    
    return mol.GetNumHeavyAtoms()

In [ ]:
posit_raw_df = posit_raw.dataframe

In [ ]:
size_dict = {}
mols = []
sizes = []
names = []
counts = []
legends = []
for i in range(19):
    scaffold_smiles = posit_raw_df[posit_raw_df.cluster_id == i].groupby(["Query_Ligand"]).head(1).scaffold_smarts.unique()[0]
    mol = Chem.MolFromSmiles(scaffold_smiles)
    names.append(f"Scaffold_{i}")
    size_dict[i] = mol.GetNumHeavyAtoms()
    sizes.append(mol.GetNumHeavyAtoms())
    counts.append(count_dict[str(i)])
    mols.append(mol)
    legends.append(f"Scaffold_{i+1} # Molecules: {count_dict[str(i)]} # Atoms: {mol.GetNumHeavyAtoms()}")
    draw_scaffold(mol, figpath / f"scaffold_{i}.png")

In [ ]:
moldf = pd.DataFrame({"Mol": mols, "Name": names, "Counts": counts, "Size": sizes, "Legend":legends})

In [ ]:
opts = Draw.MolDrawOptions()
opts.legendFraction = 0.25
opts.legendFontSize = 18
img = Draw.MolsToGridImage(moldf.Mol.tolist(), 
                           molsPerRow=4, 
                           legends=moldf.Legend.tolist(), 
                           subImgSize=(400, 200),
                           drawOptions=opts,
                           useSVG=True)
with open(figpath / "scaffold_grid.svg", 'w') as f:
    f.write(img.data)

In [ ]:
img = Draw.MolsToGridImage(moldf.Mol.tolist(), 
                           molsPerRow=4,  
                           subImgSize=(400, 200),
                           drawOptions=opts,
                           useSVG=True)
with open(figpath / "scaffold_grid_no_legend.svg", 'w') as f:
    f.write(img.data)

In [ ]:
sorted_clusters = sorted(size_dict.items(), key=lambda x: x[1], reverse=True)
cluster_order = [x[0] for x in sorted_clusters]

In [ ]:
heatmap_dfs = {
    "_".join(name): group for name, group in sdf.groupby(["Score"])
}

In [ ]:
fig_name = "scaffold_x_to_y_sorted_by_size"
for name, heatmap_df in heatmap_dfs.items():
    # Sort clusters by size
    pivot_fraction = heatmap_df.pivot(
        index="qint", columns="rint", values="Fraction"
    )
    
    # Reorder columns and index according to size
    pivot_fraction = pivot_fraction.reindex(columns=cluster_order)
    pivot_fraction = pivot_fraction.reindex(cluster_order)
    
    ref_counts = (
        heatmap_df.sort_values("rint")
        .groupby(ref)
        .head(1)[[ref, "Total"]]
        .to_dict(orient="records")
    )
    count_dict = {int(data[ref]): data["Total"] for data in ref_counts}

    # Create labels in size order
    ytick_labels = [
        f"$\\bf{int(cluster_id) + 1}$ ({size_dict[cluster_id]})" 
        for cluster_id in cluster_order
    ]
    xtick_labels = [
        f"$\\bf{int(cluster_id) + 1}$\n({size_dict[cluster_id]})" 
        for cluster_id in cluster_order
    ]
    
    # ytick_labels = [
    #     f"$\\bf{int(cluster_id) + 1}$" 
    #     for cluster_id in cluster_order
    # ]
    # xtick_labels = [
    #     f"$\\bf{int(cluster_id) + 1}$" 
    #     for cluster_id in cluster_order
    # ]
    
    plt.figure(figsize=LARGE_FIG_SIZE)

    # Create custom annotation array
    annotations = pivot_fraction.copy()
    annotations = annotations.map(lambda x: '' if x in [0.0] else f'{x:.1f}')
    
    heatmap = sns.heatmap(
        data=pivot_fraction,
        xticklabels=xtick_labels,
        yticklabels=ytick_labels,
        annot=annotations,
        fmt='',
        cmap="coolwarm_r",
        vmin=0, 
        vmax=1
    )
    # Add colorbar label
    heatmap.collections[0].colorbar.set_label(Y_LABEL, fontsize=FONT_SIZES["legend_title"], fontweight='bold')

    # Rotate axis labels for better readability
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)

    # Invert y-axis to put 0 at bottom
    plt.gca().invert_yaxis()

    # Set axis labels
    plt.xlabel(
        f"$\\bf{{Reference}}$ $\\bf{{Scaffold}}$ $\\bf{{ID}}$ \n (# Heavy Atoms in Scaffold)",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="normal",
    )
    plt.ylabel(
        f"$\\bf{{Query}}$ $\\bf{{Scaffold}}$ $\\bf{{ID}}$ \n (# Heavy Atoms in Scaffold)",
        fontsize=FONT_SIZES["ylabel"],
        fontweight="normal",
    )
    # plt.xlabel(
    #     f"Reference Scaffold ID",
    #     fontsize=FONT_SIZES["xlabel"],
    #     fontweight="bold",
    # )
    # plt.ylabel(
    #     f"Query Scaffold ID",
    #     fontsize=FONT_SIZES["ylabel"],
    #     fontweight="bold",
    # )
    plt.title(
        f"Scored by {get_label(name)}",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="bold",
    )

    plt.savefig(figpath / f"{fig_name}_{name}.svg", format="svg", bbox_inches="tight")
    plt.savefig(figpath / f"{fig_name}_{name}.png", format="png", bbox_inches="tight")

# Plot not x to x and vice versa

In [ ]:
x_to_not_x = pdf[pdf.Scaffold_Split_Option == "x_to_not_x"]

In [ ]:
# replace brackets in the query and ref columns
query = "Query_Scaffold_ID_Subset"
ref = "Reference_Scaffold_ID_Subset"
x_to_not_x[query] = (
    x_to_not_x[query]
    .astype(str)
    .apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
)
x_to_not_x["qint"] = x_to_not_x[query].astype(float).tolist()

In [ ]:
# sort 
x_to_not_x = x_to_not_x.sort_values(["qint", "Score"], ascending=[True, False])

jitter = 0.4
mult_factor = 2

# manual jitter to separate by score
jitter_vector = x_to_not_x.Score.apply(lambda x: jitter if x == "POSIT_Probability" else -jitter)
x_to_not_x["qint_jittered"] = mult_factor * x_to_not_x['qint'] + jitter_vector

xvals = list(mult_factor * x_to_not_x['qint'])
# linevals = [xval - 1.25 for xval in xvals + [xvals[-1] + mult_factor]]
linevals = [xval - 1 for xval in xvals[2:]]


fig = plt.figure(figsize=(12,6))
g = sns.scatterplot(x_to_not_x, x="qint_jittered", y="Fraction", hue="Score")

g.set(xticks=xvals,
      xticklabels=x_to_not_x[query])
# Add error bars using Matplotlib's errorbar
plt.errorbar(x=x_to_not_x["qint_jittered"], y=x_to_not_x['Fraction'], yerr=(x_to_not_x['Error_Lower'], x_to_not_x['Error_Upper']),  
             fmt='none',  # Remove the default connecting line/markers
             capsize=5,  # Adjust the size of the error bar caps
             color='black',  # Set the color of the error bars
             alpha=0.7,  # Adjust the transparency of the error bars
             elinewidth=1 # Adjust the thickness of the error bars
            )

for lineval in linevals:
    # Add vertical line at specific x position
    plt.axvline(x=lineval, color='black', linestyle='--', lw=0.5)

fig = update_labels(fig, label_map, x_label="Query Ligand Scaffold ID")

In [ ]:
save_figure(fig, figpath / "x_to_not_x" )

# Plot Similarity Split

In [ ]:
pdfsim = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/analyzed_results/all_evals_posit_combined_results.csv")
simdf = pdfsim[pdfsim["PairwiseSplit"] == "SimilaritySplit"]
fdfsim = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/analyzed_results/all_evals_fred_combined_results.csv")
fsimdf = fdfsim[fdfsim["PairwiseSplit"] == "SimilaritySplit"]
fsimdf["Method"] = "FRED"
simdf["Method"] = "POSIT"
simdf = pd.concat([simdf, fsimdf])

In [ ]:
simdf.nunique()

In [ ]:
sns.lineplot(simdf, x="Similarity_Threshold", y="Fraction", hue="Method", style="Score")

In [ ]:
simmax = simdf.groupby(["Similarity_Threshold", "Method", "Score"]).max().reset_index()

In [ ]:
sns.lineplot(simmax, x="Similarity_Threshold", y="Fraction", hue="Method", style="Score")

In [ ]:
plt = plot_filled_in_error_bars( raw_df=simmax, style_var="Score", color_var="Method", x_var="Similarity_Threshold",)
label_map.update({"Method": "Docking Algorithm", "POSIT": "OpenEye POSIT Docker", "FRED": "OpenEye FRED Docker"})
plt = update_labels(plt, label_map, x_label="TanimotoCombo Similarity", legend_subtitles=["Method", "Score"])
save_figure(plt, figpath / "similarity")

# sim split myself

In [ ]:
from harbor.analysis import cross_docking as cd
from importlib import reload
reload(cd)
sim_split = cd.EvaluatorFactory(name="increasing_similarity_tanimoto_combo_aligned")
sim_split.pairwise_split_settings.use = True
sim_split.pairwise_split_settings.similarity_split_settings.use = True
sim_split.pairwise_split_settings.similarity_split_settings.include_similar = False
sim_split.pairwise_split_settings.similarity_split_settings.similarity_groupby_dict = {
    "Type": "TanimotoCombo",
    "Aligned": True,
}
sim_split.n_bootstraps = 10
evs = sim_split.create_evaluators(posit_raw)

In [ ]:
len(evs)

In [ ]:
results_df = cd.Results.df_from_results(cd.Results.calculate_results(posit_raw, evs))

In [ ]:
sns.lineplot(results_df, x="Similarity_Threshold", y="Fraction", hue="Score")

In [ ]:
from harbor.analysis import cross_docking as cd
from importlib import reload
reload(cd)
sim_split = cd.EvaluatorFactory(name="increasing_similarity_tanimoto_combo_aligned")
sim_split.pairwise_split_settings.use = True
sim_split.pairwise_split_settings.similarity_split_settings.use = True
sim_split.pairwise_split_settings.similarity_split_settings.include_similar = False
sim_split.pairwise_split_settings.similarity_split_settings.similarity_groupby_dict = {
    "Type": "MCS",
    
}
sim_split.n_bootstraps = 10
evs = sim_split.create_evaluators(posit_raw)

In [ ]:
len(evs)

In [ ]:
results_df = cd.Results.df_from_results(cd.Results.calculate_results(posit_raw, evs))

In [ ]:
sns.lineplot(results_df, x="Similarity_Threshold", y="Fraction", hue="Score")

In [ ]:
evs[0]

In [ ]:
from harbor.analysis import cross_docking as cd
from importlib import reload
reload(cd)
sim_split = cd.EvaluatorFactory(name="increasing_similarity_tanimoto_combo_aligned")
sim_split.pairwise_split_settings.use = True
sim_split.pairwise_split_settings.similarity_split_settings.use = True
sim_split.pairwise_split_settings.similarity_split_settings.include_similar = False
sim_split.pairwise_split_settings.similarity_split_settings.update_reference_settings.use = True
sim_split.pairwise_split_settings.similarity_split_settings.update_reference_settings.use_logarithmic_scaling = True
sim_split.pairwise_split_settings.similarity_split_settings.similarity_groupby_dict = {
    "Type": "MCS",
    
}
sim_split.pairwise_split_settings.similarity_split_settings.similarity_n_thresholds = 5
sim_split.n_bootstraps = 1
evs = sim_split.create_evaluators(posit_raw)

In [ ]:
len(evs)

In [ ]:
results_df = cd.Results.df_from_results(cd.Results.calculate_results(posit_raw, evs))

In [ ]:
sns.lineplot(results_df, x="Similarity_Threshold", y="Fraction", hue="N_Reference_Structures", style="Score")
plt.save

In [ ]:
evs[-1]